# 图 9–10 · 流程与基线 SVG 素材

选择 `PPG_sensor_env`，从上至下运行。不需要样本 CSV 或分析结果。
生成的图为**示意图**，使用可编辑中文文字与内联矢量图形。

## 只改这里
下方单元格集中控制所有文案、配色和输出目录。`MEANING_CARDS` 的标题、正文可直接输入中文；
增加或删除列表项即可增删说明卡片，最多每行三张，长文案自动换行并增高。
前三张卡片按顺序与信号分工、物理对应、佩戴应用连接；额外卡片为补充解释。
流程节点建议使用短标题；意义卡片适合较长说明。

In [ ]:
# 从仓库根目录或 figures/ 启动均可；只导入本工作树的模块。
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "udeco_uh_roi_edge_detection.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从仓库根目录或 figures/ 目录启动 Jupyter。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path

# ===== 只改这里：文字均可直接编辑，不需要修改 SVG 代码 =====
OUTPUT_DIR = PROJECT_ROOT / "figures" / "assets"
FLOW_TEXT = {
    "title": "从扰动定位到佩戴状态理解",
    "subtitle": "Uh 引导的 Udeco 状态识别 · 方法与应用示意",
    "algorithm": "双分支识别", "meaning": "物理解释与应用意义",
    "uh_lane": "Uh · 扰动线索", "udeco_lane": "Udeco · 接触线索",
    "acc": "ACC 定位运动段", "application_link": "应用指向",
    "input": {"title": "读取与对齐", "note": "Uh / Uc / ACC"},
    "uh": {"title": "滤波去基线", "note": "4 点中值 · LOWESS"},
    "frequency": {"title": "主频估计", "note": "运动段 · 幅频谱"},
    "roi": {"title": "检峰与 ROI", "note": "定位扰动时段"},
    "calibration": {"title": "标定计算", "note": "Uh、Uc → Udeco"},
    "udeco": {"title": "滤波去基线", "note": "4 点中值 · LOWESS"},
    "edges": {"title": "ROI 内检沿", "note": "边沿补全"},
    "states": {"title": "状态识别", "note": "接触事件表征"},
    "state_labels": ["脱离—贴合", "释放—施力", "滑动"],
    "footer": "波形与腕表示意；物理解释及应用指向不等同于已验证的完整模式分类器。",
}

# 每张卡片只需 title 和 lines；按顺序对应“信号分工、物理对应、佩戴应用”。
# 增删列表项可增删卡片；字符串中的 \n 可以强制换行。
MEANING_CARDS = [
    {"title": "信号分工", "lines": [
        "Uh：定位扰动时段", "仅凭 Uh 难以区分接触变化。", "Udeco：补充接触状态信息。"]},
    {"title": "物理对应", "lines": [
        "脱离—贴合：接触变化", "释放—施力：受力变化", "滑动保留为当前展示标签。"]},
    {"title": "佩戴应用", "lines": [
        "可用于腕表佩戴状态识别", "可辅助佩戴与交互模式区分。"]},
]
FLOW_STYLE = {
    "width": 1560,       # 输出显示宽度；内部流程采用固定逻辑坐标并整体等比缩放
    "font_family": "Microsoft YaHei, Noto Sans CJK SC, SimHei, sans-serif",
    "title_size": 34, "node_title_size": 22, "note_size": 17,
    "card_title_size": 24, "body_size": 21, "line_height": 1.6,
    "card_gap": 24, "card_padding": 26, "card_columns": 3,
    "ink": "#233742", "muted": "#647581", "border": "#D9E3E7",
    "uh": "#347C9D", "uh_fill": "#F0F7FA",
    "udeco": "#388E82", "udeco_fill": "#EFF8F5",
    "edge": "#B57952", "edge_fill": "#FCF5EE",
    "states": ["#DAA083", "#88B5CE", "#BCBEC3"],
}


## SVG 生成函数

以下函数仅使用 Python 标准库。修改文字无需编辑这一单元格。
`build_flowchart_svg(FLOW_TEXT, MEANING_CARDS, FLOW_STYLE)` 返回 SVG 字符串，不写文件。
中文字体优先使用微软雅黑，并提供字体回退；SVG 文字保留为 `<text>`。

In [ ]:
import math
import unicodedata
from html import escape
from xml.etree import ElementTree as ET


def wrap_text(value, width, size):
    """Wrap CJK and Latin text; preserve explicit newlines and blank lines."""
    if width <= 0 or size <= 0:
        raise ValueError("文本宽度和字号必须为正数。")
    result = []
    for paragraph in str(value).split("\n"):
        line, used = "", 0.0
        for char in paragraph:
            units = 1.0 if unicodedata.east_asian_width(char) in "WF" else .62
            if char.isspace():
                units = .35
            advance = units * size
            if line and used + advance > width:
                result.append(line)
                line, used = "", 0.0
            line += char
            used += advance
        result.append(line)
    return result


def build_flowchart_svg(flow_text, meaning_cards, style):
    """Pure renderer. Returns self-contained SVG text; does not write files."""
    s, f = style, flow_text
    required = ("width", "title_size", "node_title_size", "note_size", "card_title_size",
                "body_size", "line_height", "card_gap", "card_padding")
    if any(not math.isfinite(float(s[k])) or float(s[k]) <= 0 for k in required):
        raise ValueError("宽度、字号、间距和行高必须为有限正数。")
    if len(f["state_labels"]) != 3 or len(s["states"]) != 3:
        raise ValueError("状态标签与状态颜色均需保持三项。")
    cols = max(1, min(3, int(s["card_columns"]), len(meaning_cards) or 1))
    canvas_w, margin = 1560, 48
    gap, pad = float(s["card_gap"]), float(s["card_padding"])
    available = canvas_w - margin * 2
    body_size, line_h = float(s["body_size"]), float(s["body_size"]) * s["line_height"]
    card_w = (available - gap * (cols - 1)) / cols
    text_w = card_w - pad * 2 - 18
    if text_w < body_size * 4:
        raise ValueError("卡片内边距或间距过大；请减少 card_padding/card_gap。")

    # Measure first. Cards grow with their content; each row shares its tallest height.
    layouts, y = [], 780.0
    for start in range(0, len(meaning_cards), cols):
        row = []
        for offset, card in enumerate(meaning_cards[start:start + cols]):
            if not isinstance(card.get("lines"), (list, tuple)):
                raise ValueError("每张说明卡片的 lines 必须是字符串列表。")
            title_lines = wrap_text(card["title"], card_w - pad * 2 - 60, s["card_title_size"])
            body_lines = [wrap_text(item, text_w, body_size) for item in card["lines"]]
            title_h = len(title_lines) * s["card_title_size"] * 1.35
            body_h = sum(len(lines) * line_h + 12 for lines in body_lines)
            height = max(220, pad * 2 + title_h + 28 + body_h)
            row.append(dict(index=start + offset, x=margin + offset * (card_w + gap), y=y,
                            w=card_w, h=height, title=title_lines, body=body_lines, title_h=title_h))
        row_h = max(item["h"] for item in row)
        for item in row:
            item["h"] = row_h
        layouts.extend(row)
        y += row_h + gap
    footer_lines = wrap_text(f["footer"], available, s["note_size"])
    footer_y = (y if layouts else 765) + 16
    canvas_h = footer_y + len(footer_lines) * s["note_size"] * 1.5 + 34
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{s["width"]}" '
             f'height="{canvas_h * s["width"] / canvas_w:.2f}" viewBox="0 0 {canvas_w} {canvas_h:.2f}" '
             f'role="img" aria-labelledby="flow-title flow-desc">',
             f'<title id="flow-title">{escape(f["title"])}</title>',
             '<desc id="flow-desc">双分支算法与物理解释、腕表佩戴应用的对应示意。</desc>',
             '<defs><marker id="arrow" viewBox="0 0 10 10" refX="9" refY="5" '
             'markerWidth="6" markerHeight="6" orient="auto-start-reverse">'
             '<path d="M 0 0 L 10 5 L 0 10 z" fill="#8B9DA5"/></marker></defs>',
             f'<rect width="1560" height="{canvas_h:.2f}" fill="white"/>',
             f'<g font-family="{escape(s["font_family"], quote=True)}">']

    def rect(x, y, w, h, fill, stroke=None, radius=18):
        parts.append(f'<rect x="{x:.2f}" y="{y:.2f}" width="{w:.2f}" height="{h:.2f}" '
                     f'rx="{radius}" fill="{fill}" stroke="{stroke or s["border"]}" stroke-width="1.2"/>')

    def text(x, y, value, size=None, color=None, weight=400, anchor="start"):
        # One text element per line keeps text directly editable in vector editors.
        parts.append(f'<text x="{x:.2f}" y="{y:.2f}" font-size="{size or s["note_size"]}" '
                     f'fill="{color or s["ink"]}" font-weight="{weight}" text-anchor="{anchor}">'
                     f'{escape(str(value))}</text>')

    def path(d, color=None, width=2, dashed=False, arrow=False):
        parts.append(f'<path d="{d}" fill="none" stroke="{color or s["border"]}" '
                     f'stroke-width="{width}" stroke-linecap="round" stroke-linejoin="round" '
                     + ('stroke-dasharray="6 6" ' if dashed else '')
                     + ('marker-end="url(#arrow)" ' if arrow else '') + '/>')

    def route(points, **kwargs):
        path("M " + " L ".join(f"{x:.2f} {y:.2f}" for x, y in points), **kwargs)

    def icon(kind, x, y, w, h, color):
        """Simple vectors, all schematic; no raster resources or external hrefs."""
        if kind == "watch":
            rect(x + w * .36, y, w * .28, h, "#F5F8FA", color, 7)
            rect(x + w * .2, y + h * .23, w * .6, h * .54, "white", color, 10)
            route([(x + w * .3, y + h * .52), (x + w * .44, y + h * .63),
                   (x + w * .7, y + h * .39)], color=color, width=2)
            return
        if kind == "input":
            for j in range(3):
                rect(x + 4 * j, y + h * .24 * j, w * .83, h * .32, "white", color, 7)
                path(f'M {x + 14 + 4*j} {y + h*.24*j + 10} h {w*.45}', color, 2)
            return
        route([(x, y + h), (x + w, y + h)], color="#D0DCE1", width=1)
        if kind == "frequency":
            for i, value in enumerate([.12, .23, .4, .96, .57, .2, .1, .08]):
                xx = x + (i + .5) * w / 8
                path(f'M {xx} {y+h} V {y+h-value*h}', color, 4)
            return
        if kind == "calibration":
            text(x + w / 2, y + h * .36, "Uh − Uc", 20, color, anchor="middle")
            route([(x + w * .17, y + h * .53), (x + w * .83, y + h * .53)], color=color, width=1.5)
            text(x + w / 2, y + h * .91, "c − Uc", 20, color, anchor="middle")
            return
        if kind == "edges":
            rect(x + .19*w, y, .62*w, h, "#EAF1F3", "none", 5)
            route([(x, y+.82*h), (x+.2*w, y+.82*h), (x+.37*w, y+.16*h),
                   (x+.65*w, y+.16*h), (x+.82*w, y+.82*h), (x+w, y+.82*h)], color=color, width=2.5)
            route([(x+.2*w, y+.82*h), (x+.37*w, y+.16*h)], color="#B8503E", width=4)
            route([(x+.65*w, y+.16*h), (x+.82*w, y+.82*h)], color="#2769A0", width=4)
            return
        if kind == "roi":
            for a in (.13, .57):
                rect(x + a*w, y, .25*w, h, "#DFEAF0", "none", 5)
        points = []
        for i in range(101):
            u = i / 100
            value = .58 - .37 * math.sin(4 * math.pi * u) ** 7
            points.append((x + w*u, y + h*value))
        route(points, color=color, width=2.4)
        if kind == "roi":
            for u in (.125, .625):
                parts.append(f'<circle cx="{x+w*u}" cy="{y+h*.21}" r="4" fill="{color}"/>')
        else:
            route([(x, y+h*.68), (x+w*.35, y+h*.51), (x+w*.7, y+h*.6), (x+w, y+h*.43)],
                  color="#B0BCC3", width=1.5, dashed=True)

    def node(key, x, y, w, h, kind, tone):
        rect(x, y, w, h, s[tone + "_fill"], s["border"])
        text(x + 18, y + 34, f[key]["title"], s["node_title_size"], s[tone], 600)
        icon(kind, x + 28, y + 56, w - 56, h - 103, s[tone])
        text(x + 18, y + h - 16, f[key]["note"], s["note_size"], s["muted"])

    text(48, 60, f["title"], s["title_size"], weight=700)
    text(48, 95, f["subtitle"], 18, s["muted"])
    text(48, 145, "01", 18, s["uh"], 700)
    text(86, 145, f["algorithm"], 23, weight=600)
    text(250, 186, f["uh_lane"], 18, s["uh"], 600)
    text(250, 445, f["udeco_lane"], 18, s["udeco"], 600)
    node("input", 48, 330, 158, 166, "input", "uh")
    node("uh", 250, 210, 210, 170, "wave", "uh")
    node("frequency", 490, 210, 210, 170, "frequency", "uh")
    node("roi", 730, 210, 210, 170, "roi", "uh")
    node("calibration", 250, 470, 210, 170, "calibration", "udeco")
    node("udeco", 490, 470, 450, 170, "wave", "udeco")
    node("edges", 1008, 330, 208, 210, "edges", "edge")
    rect(1260, 300, 252, 270, "#FAFBFC")
    text(1282, 335, f["states"]["title"], s["node_title_size"], weight=600)
    text(1282, 362, f["states"]["note"], s["note_size"], s["muted"])
    for j, (label, color) in enumerate(zip(f["state_labels"], s["states"])):
        rect(1280, 385 + 52*j, 210, 40, color, "none", 9)
        text(1385, 412 + 52*j, label, 20, anchor="middle")
    # ACC is an interval cue, not a third classification branch.
    rect(490, 163, 210, 30, "#F5F7F8", "none", 10)
    text(595, 184, f["acc"], 16, s["muted"], anchor="middle")
    route([(595, 193), (595, 207)], arrow=True)
    for points in [
        [(206, 413), (226, 413), (226, 295), (247, 295)],
        [(226, 413), (226, 555), (247, 555)],
        [(460, 295), (487, 295)], [(700, 295), (727, 295)],
        [(460, 555), (487, 555)],
        [(940, 295), (974, 295), (974, 410), (1005, 410)],
        [(940, 555), (974, 555), (974, 460), (1005, 460)],
        [(1216, 435), (1257, 435)],
    ]:
        route(points, color="#8B9DA5", width=1.8, arrow=True)

    text(48, 730, "02", 18, s["udeco"], 700)
    text(86, 730, f["meaning"], 23, weight=600)
    # The first three editable cards correspond to the three semantic connections.
    anchors = [(355, 640), (1112, 540), (1386, 570)]
    for item, (sx, sy) in zip(layouts[:3], anchors):
        tx = item["x"] + item["w"] * .64
        route([(sx, sy + 4), (sx, 680), (tx, 680), (tx, item["y"] - 8)],
              color="#AFBCC3", width=1.5, dashed=True, arrow=True)
        if item["index"] == 2:
            text(sx + 12, 662, f["application_link"], 16, s["muted"])
    for item in layouts:
        i, x, y, w, h = (item[k] for k in ("index", "x", "y", "w", "h"))
        color = [s["uh"], s["edge"], s["udeco"]][i % 3]
        parts.append(f'<g data-card-index="{i}" data-card-height="{h:.2f}">')
        rect(x, y, w, h, "#F8FAFB")
        rect(x + pad, y + 24, 5, 30, color, "none", 2)
        for j, line in enumerate(item["title"]):
            text(x + pad + 17, y + pad + s["card_title_size"] + j * s["card_title_size"] * 1.35,
                 line, s["card_title_size"], weight=600)
        if i == 2:
            icon("watch", x + w - pad - 42, y + pad - 2, 42, 52, color)
        else:
            text(x + w - pad, y + pad + 23, f"{i+1:02d}", 18, color, weight=600, anchor="end")
        line_y = y + pad + item["title_h"] + 34
        for wrapped in item["body"]:
            parts.append(f'<circle cx="{x+pad+3}" cy="{line_y-body_size*.32}" r="2.6" fill="{color}"/>')
            for line in wrapped:
                text(x + pad + 18, line_y, line, body_size)
                line_y += line_h
            line_y += 12
        parts.append('</g>')
    for j, line in enumerate(footer_lines):
        text(48, footer_y + j * s["note_size"] * 1.5, line, s["note_size"], s["muted"])
    parts.extend(['</g>', '</svg>'])
    svg = "\n".join(parts)
    ET.fromstring(svg)
    return svg


## 生成文件并预览

In [ ]:
from IPython.display import SVG, display

svg_text = build_flowchart_svg(FLOW_TEXT, MEANING_CARDS, FLOW_STYLE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
svg_path = OUTPUT_DIR / "udeco_flowchart_cn.svg"
svg_path.write_text(svg_text, encoding="utf-8")
display(SVG(data=svg_text))
print(f"已生成：{svg_path}")

## 如何修改

1. 修改顶部 `FLOW_TEXT` 中的节点文字，或 `MEANING_CARDS` 中的 `title` / `lines`。
2. 运行配置、函数、生成三个代码单元格，再到主 notebook 重新运行图 9。
3. 正文字符串中使用 `\n` 可强制换行；也可添加新的 `lines` 条目。
4. 配色、字号、间距在 `FLOW_STYLE` 中修改；`width` 控制整个 SVG 的显示宽度。
5. 移动文件时，同步修改主 notebook 图 9 的 `FLOW9_SVG_PATH`。

直接编辑 SVG 也可以修改文字，但下次运行生成单元格会覆盖同名文件，建议始终修改配置。
`slide` 是原算法 `cancelled` 的展示别名，不代表已独立验证的滑动识别。
下层卡片为物理解释及应用指向，不表示已经实现完整腕表模式分类器。

## 图 10 · LOWESS 基线估计说明（21:6 SVG）

LOWESS 是局部加权回归平滑。这里展示**稳健 LOWESS 估计基线**，目的不是直接滤除接触尖峰。
三个面板使用同一合成数据和相同中值预处理；曲线由实际计算生成。
低通采用零相位 Butterworth，避免把差异误归因于相位延迟。所选截止频率与 LOWESS 窗口并非严格等效带宽，图示不是一般性能排名。

### 只改这里：文字、配色与示意参数
本节可独立执行，无需运行图 9 或读取样本。依次运行配置、函数和生成单元格。

In [ ]:
# 从仓库根目录或 figures/ 启动均可；只导入本工作树的模块。
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "udeco_uh_roi_edge_detection.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从仓库根目录或 figures/ 目录启动 Jupyter。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path

# 图 10 可独立运行：依次运行本配置、函数、生成三个单元格。
LOWESS_OUTPUT_DIR = PROJECT_ROOT / "figures" / "assets"
LOWESS_TEXT = {
    "title": "稳健 LOWESS：从多尖峰波形中估计慢变基线",
    "subtitle": "同一合成波形与中值预处理 · 估计慢变基线，供后续去基线与边沿识别使用。",
    "panels": ["多尖峰波形", "常规低通", "稳健 LOWESS"],
    "captions": ["目标：分离慢变基线与短时尖峰", "尖峰含低频成分，仍可拖动基线", "降低大残差点权重，更贴近慢趋势"],
    "meaning_title": "为何适合此类基线估计？",
    "meaning_points": [
        ["局部拟合", "随时间跟踪缓慢漂移的基线"],
        ["残差降权", "减小孤立大残差尖峰对拟合的牵引"],
        ["基线相减", "减去慢趋势，再分析尖峰与边沿"],
    ],
    "legend": ["输入波形", "已知基线", "低通估计", "LOWESS 估计"],
    "time_axis": "时间（s）",
    "footer": "合成示意，非等效带宽性能评测。适用：尖峰在局部窗口内占比较低；宽峰、密集峰或真实台阶需另行验证。",
}
LOWESS_STYLE = {
    "width": 2100,    # 高度固定为 width * 6 / 21；文字在逻辑画布内整体缩放
    "font_family": "Microsoft YaHei, Noto Sans CJK SC, SimHei, sans-serif",
    "title_size": 32, "body_size": 21, "caption_size": 20,
    "ink": "#263C47", "muted": "#697C87", "border": "#DCE6EB",
    "signal": "#91A0AC", "baseline": "#4B5A65", "lowpass": "#BD8257", "lowess": "#2C8A7B",
}
LOWESS_DEMO = {
    "fs": 40, "duration_sec": 60.0, "median_points": 4,
    "window_sec": 10.0, "robust_iters": 1,
    "lowpass_order": 4, "cutoff_hz": 0.15,
    "peak_centres_sec": [15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45],
    "peak_width_sec": 0.18,    # 高斯尖峰的标准差，不是 FWHM
}


### 合成波形、基线估计与 SVG 函数

使用已有 NumPy、SciPy、statsmodels 依赖。SVG 保留中文 `<text>`，宽高比固定 21:6。
段落文字超过预留行数时明确报错，不截断文字；标题建议保持精简。

In [ ]:
from html import escape
import unicodedata
from xml.etree import ElementTree as ET
import numpy as np
from scipy.ndimage import median_filter
from scipy.signal import butter, sosfiltfilt
from statsmodels.nonparametric.smoothers_lowess import lowess


def make_lowess_demo(config):
    """Deterministic synthetic example; both estimates use the same median-cleaned input."""
    c = config
    for key in ("fs", "duration_sec", "window_sec", "cutoff_hz", "peak_width_sec"):
        if not np.isfinite(c[key]) or c[key] <= 0:
            raise ValueError(f"{key} 必须为有限正数。")
    for key in ("median_points", "lowpass_order", "robust_iters"):
        if not isinstance(c[key], (int, np.integer)) or c[key] < (0 if key == "robust_iters" else 1):
            raise ValueError(f"{key} 必须为有效整数。")
    if not 0 < c["cutoff_hz"] < c["fs"] / 2:
        raise ValueError("低通截止频率必须低于 Nyquist 频率。")
    if not 0 < c["window_sec"] <= c["duration_sec"]:
        raise ValueError("LOWESS 窗口不能超过总时长。")
    if c["robust_iters"] < 1:
        raise ValueError("本图说明稳健 LOWESS，robust_iters 至少为 1。")
    n = round(c["fs"] * c["duration_sec"])
    if n < 32:
        raise ValueError("示意数据至少需要 32 个采样点。")
    t = np.arange(n) / c["fs"]
    baseline = .14 + .0015 * t + .045 * np.sin(t / 11)
    raw = baseline + .005 * np.sin(13 * t)
    for j, centre in enumerate(c["peak_centres_sec"]):
        if not np.isfinite(centre) or not 0 < centre < c["duration_sec"]:
            raise ValueError("尖峰中心必须在记录时长内。")
        raw += (.8 + .22 * np.cos(j)) * np.exp(-.5 * ((t - centre) / c["peak_width_sec"]) ** 2)
    clean = median_filter(raw, size=c["median_points"], mode="nearest")
    fraction = min(1., max(3. / n, c["window_sec"] / (n / c["fs"])))
    robust = lowess(clean, t, frac=fraction, it=c["robust_iters"], return_sorted=False)
    sos = butter(c["lowpass_order"], c["cutoff_hz"], fs=c["fs"], output="sos")
    filtered = sosfiltfilt(sos, clean)
    return dict(time=t, raw=raw, clean=clean, baseline=baseline, lowess=robust, lowpass=filtered)


def build_lowess_svg(labels, style, demo, config):
    """Create an editable, self-contained SVG with exactly 21:6 aspect ratio."""
    if len(labels["panels"]) != 3 or len(labels["captions"]) != 3 or len(labels["meaning_points"]) != 3:
        raise ValueError("图 10 固定为三个示意面板和三条原理说明。")
    if len(labels["legend"]) != 4:
        raise ValueError("图例需保留四项。")
    s = style
    for key in ("width", "title_size", "body_size", "caption_size"):
        if not np.isfinite(s[key]) or s[key] <= 0:
            raise ValueError("尺寸和字号必须为有限正数。")
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{s["width"]}" '
             f'height="{s["width"]*6/21:.3f}" viewBox="0 0 2100 600" role="img" aria-labelledby="lowess-title">',
             f'<title id="lowess-title">{escape(labels["title"])}</title>',
             '<rect width="2100" height="600" fill="white"/>',
             f'<g font-family="{escape(s["font_family"], quote=True)}">']

    def wrap(value, width, size):
        lines = []
        for paragraph in str(value).split("\n"):
            current, used = "", 0.
            for char in paragraph:
                advance = size * (1. if unicodedata.east_asian_width(char) in "WF" else .62)
                if current and used + advance > width:
                    lines.append(current)
                    current, used = "", 0.
                current += char
                used += advance
            lines.append(current)
        return lines

    def text(x, y, value, size=20, color=None, weight=400, anchor="start"):
        parts.append(f'<text x="{x}" y="{y}" font-size="{size}" fill="{color or s["ink"]}" '
                     f'font-weight="{weight}" text-anchor="{anchor}">{escape(str(value))}</text>')

    def block(x, y, value, width, size, max_lines=2, color=None):
        lines = wrap(value, width, size)
        if len(lines) > max_lines:
            raise ValueError(f"21:6 画布内文案过长，请精简或减小字号：{value}")
        for j, line in enumerate(lines):
            text(x, y + j * size * 1.4, line, size, color)

    def rect(x, y, w, h, fill):
        parts.append(f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="16" '
                     f'fill="{fill}" stroke="{s["border"]}" stroke-width="1"/>')

    def polyline(points, color, width=2, opacity=1., dash=False):
        coords = " ".join(f"{x:.2f},{y:.2f}" for x, y in points)
        parts.append(f'<polyline points="{coords}" fill="none" stroke="{color}" '
                     f'stroke-width="{width}" opacity="{opacity}" stroke-linejoin="round" '
                     f'stroke-linecap="round"' + (' stroke-dasharray="7 5"' if dash else '') + '/>')

    text(48, 57, labels["title"], s["title_size"], weight=700)
    block(48, 96, labels["subtitle"], 1998, 20, max_lines=1, color=s["muted"])
    time = demo["time"]
    all_values = np.concatenate([demo[k] for k in ("clean", "baseline", "lowpass", "lowess")])
    lo, hi = float(all_values.min()), float(all_values.max())
    padding = max(.03, (hi - lo) * .08)
    lo, hi = lo - padding, hi + padding
    details = [f'{config["median_points"]} 点中值预处理 · 同一输入',
               f'{config["lowpass_order"]} 阶 Butterworth · {config["cutoff_hz"]:g} Hz · 零相位',
               f'{config["window_sec"]:g} s 窗口 · {config["robust_iters"]} 次稳健迭代']
    for index, (name, caption, x0) in enumerate(zip(labels["panels"], labels["captions"], (48, 528, 1008))):
        rect(x0, 128, 452, 366, "#FAFCFD")
        text(x0 + 22, 168, f'{"abc"[index]}  {name}', 24, weight=600)
        text(x0 + 22, 198, details[index], 17, s["muted"])
        px, py, pw, ph = x0 + 26, 222, 400, 162
        def curve(values):
            return zip(px + (time / time[-1]) * pw, py + ph - (np.asarray(values) - lo) / (hi - lo) * ph)
        polyline([(px, py + ph + 8), (px + pw, py + ph + 8)], s["border"], 1)
        polyline(curve(demo["clean"]), s["signal"], 1.6, .95 if index == 0 else .45)
        polyline(curve(demo["baseline"]), s["baseline"], 2, dash=True)
        if index:
            key = "lowpass" if index == 1 else "lowess"
            polyline(curve(demo[key]), s[key], 3.2)
        for fraction in (0, .5, 1):
            at = px + pw * fraction
            polyline([(at, py + ph + 8), (at, py + ph + 12)], s["border"], 1)
            text(at, 417, f'{time[-1] * fraction:.0f}', 15, s["muted"], anchor="middle")
        text(px + pw / 2, 438, labels["time_axis"], 15, s["muted"], anchor="middle")
        block(x0 + 22, 470, caption, 410, s["caption_size"], max_lines=1)
    rect(1500, 128, 552, 366, "#F0F7F5")
    text(1528, 170, labels["meaning_title"], 25, s["lowess"], weight=600)
    for i, (heading, detail) in enumerate(labels["meaning_points"]):
        y = 228 + i * 86
        text(1528, y, f'0{i+1}', 18, s["lowess"], weight=700)
        text(1571, y, heading, 23, weight=600)
        block(1571, y + 31, detail, 451, s["body_size"], max_lines=1, color=s["muted"])
    for x, label, color, dashed in zip((48, 323, 598, 873), labels["legend"],
                                      (s["signal"], s["baseline"], s["lowpass"], s["lowess"]),
                                      (False, True, False, False)):
        polyline([(x, 529), (x + 42, 529)], color, 2.6, dash=dashed)
        text(x + 54, 536, label, 19, s["muted"])
    block(48, 581, labels["footer"], 1998, 18, max_lines=1, color=s["muted"])
    parts.extend(['</g>', '</svg>'])
    svg = "\n".join(parts)
    ET.fromstring(svg)
    return svg


### 生成图 10 并预览

In [ ]:
from IPython.display import SVG, display

lowess_demo = make_lowess_demo(LOWESS_DEMO)
lowess_svg = build_lowess_svg(LOWESS_TEXT, LOWESS_STYLE, lowess_demo, LOWESS_DEMO)
LOWESS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
lowess_svg_path = LOWESS_OUTPUT_DIR / "lowess_baseline_explanation.svg"
lowess_svg_path.write_text(lowess_svg, encoding="utf-8")
display(SVG(data=lowess_svg))
print(f"已生成：{lowess_svg_path}")

### 原理及修改备注

- **为什么稳健 LOWESS 有帮助**：局部回归跟踪慢变趋势；稳健迭代按残差降低异常点的影响。窄尖峰在局部窗口中占比较低时，可减小尖峰对基线的牵引。
- **为何低通仍可能受尖峰影响**：短时尖峰具有宽频谱，其中的低频分量也会进入低通结果。线性频率滤波不等同于按残差识别并降低尖峰权重。
- 宽峰、密集尖峰、真实基线台阶或纯高频噪声场景下，不能假定 LOWESS 一定更好。需要结合窗口、稳健次数与截止频率验证；本图不改变项目处理算法。
- **修改文字**：编辑 `LOWESS_TEXT`；样式修改 `LOWESS_STYLE`；改变尖峰、窗口、稳健次数或低通参数修改 `LOWESS_DEMO`。重新生成后运行主 notebook 图 10。
- 固定 21:6 版式适合精简文案；长文案报错时应缩短文字或调小相关字号。
- 技术参考：[statsmodels LOWESS](https://www.statsmodels.org/stable/generated/statsmodels.nonparametric.smoothers_lowess.lowess.html)。